In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
import numpy as np
# 1) Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

writer = SummaryWriter(log_dir="runs/resnext29_cifar100")


Using device: cuda


In [2]:
transform_train = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=4)
test_loader  = torch.utils.data.DataLoader(testset,  batch_size=128,  shuffle=False, num_workers=4)

In [3]:
from resnext import resnext29_8x64d  # assuming you defined this from previous step

model = resnext29_8x64d(num_classes=100).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[150, 225], gamma=0.1)
epochs = 300
batch_size = 128
criterion = nn.CrossEntropyLoss().to(device)
scaler = GradScaler(init_scale=2**12, device="cuda")


In [4]:
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, correct = 0.0, 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with autocast(device_type=device.type):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    scheduler.step()

    train_acc = 100. * correct / total
    train_loss = total_loss / total

    # Validation
    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()

    val_acc = 100. * val_correct / val_total
    val_loss = val_loss / val_total

    writer.add_scalar("LR/epoch", optimizer.param_groups[0]["lr"], epoch)
    writer.add_scalars("Accuracy", {"Train": train_acc, "Validation": val_acc}, epoch)
    writer.add_scalars("Loss",     {"Train": train_loss, "Validation": val_loss}, epoch)

    print(f"Epoch {epoch:03d}: Train Loss {train_loss:.4f}, Acc {train_acc:.2f}% | Val Loss {val_loss:.4f}, Acc {val_acc:.2f}%")

Epoch 001: Train Loss 4.2492, Acc 6.25% | Val Loss 3.8461, Acc 10.00%
Epoch 002: Train Loss 3.4662, Acc 16.10% | Val Loss 3.5393, Acc 16.60%
Epoch 003: Train Loss 2.8536, Acc 27.19% | Val Loss 2.8471, Acc 27.36%
Epoch 004: Train Loss 2.3785, Acc 36.87% | Val Loss 2.5912, Acc 34.08%
Epoch 005: Train Loss 2.0677, Acc 43.60% | Val Loss 2.2777, Acc 41.75%
Epoch 006: Train Loss 1.8587, Acc 48.24% | Val Loss 2.0310, Acc 45.44%
Epoch 007: Train Loss 1.6988, Acc 52.46% | Val Loss 2.0030, Acc 46.74%
Epoch 008: Train Loss 1.5817, Acc 55.33% | Val Loss 1.8582, Acc 49.60%
Epoch 009: Train Loss 1.4989, Acc 57.38% | Val Loss 1.8356, Acc 50.92%
Epoch 010: Train Loss 1.4230, Acc 59.33% | Val Loss 2.5184, Acc 41.91%
Epoch 011: Train Loss 1.3863, Acc 60.40% | Val Loss 2.0133, Acc 48.77%
Epoch 012: Train Loss 1.3366, Acc 61.58% | Val Loss 1.8942, Acc 51.31%
Epoch 013: Train Loss 1.3007, Acc 62.48% | Val Loss 1.7836, Acc 53.38%
Epoch 014: Train Loss 1.2605, Acc 63.83% | Val Loss 2.0147, Acc 50.18%
Epoch 0

In [5]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss_sum += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    avg_loss = loss_sum / total
    accuracy = 100. * correct / total
    return avg_loss, accuracy

In [6]:
test_loss, test_acc = evaluate(model, test_loader, device)
print(f"Final Test Loss: {test_loss:.4f} | Final Test Accuracy: {test_acc:.2f}%")

Final Test Loss: 0.7094 | Final Test Accuracy: 81.93%


In [7]:
save_path = "resnext29_cifar100.pth"
torch.save(model.state_dict(), save_path)
print(f"Model weights saved to {save_path}")
sd = torch.load(save_path, map_location="cpu")
print(sd["fc.weight"].shape, sd["fc.bias"].shape)

Model weights saved to resnext29_cifar100.pth
torch.Size([100, 1024]) torch.Size([100])
